# Pipeline trích xuất metadata từ GRI standards

In [2]:
import os
import json
import re
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

import pdfplumber

from FlagEmbedding import BGEM3FlagModel

from tqdm.auto import tqdm

In [3]:
BASE_DIR = Path(r"D:\Final_GRAG")
GRI_STANDARDS_DIR = BASE_DIR / "GRI_standards"

METADATA_DIR = BASE_DIR / "metadata"
METADATA_DIR.mkdir(exist_ok=True)

# Folder cho output của module 1
UNITS_DIR = METADATA_DIR / "gri_units"
EDGES_DIR = METADATA_DIR / "gri_edges"

for dir_path in [UNITS_DIR, EDGES_DIR]:
    dir_path.mkdir(exist_ok=True)

In [4]:
pdf_files = list(GRI_STANDARDS_DIR.glob("*.pdf"))
print(len(pdf_files))

38


## PDF parser

### Xử lý tên file 

In [5]:
def parse_gri_filename(filename: str) -> Dict[str, any]:
    # Bỏ đuôi pdf
    name = filename.replace('.pdf', '')
    
    # Pattern: GRI {loại tiêu chuẩn}[_:] {tên tiêu chuẩn} {năm phát hành}
    pattern = r'GRI\s*(\d+)[_:]\s*([^0-9]+?)\s*(\d{4})'
    match = re.search(pattern, name)
    
    if match:
        standard_num = match.group(1)
        standard_name = match.group(2).strip().strip('_').strip()
        year = int(match.group(3))
        
        standard_id = f"GRI {standard_num}"
        
        # Chia các standards về các loại
        if standard_num in ['1', '2', '3']:
            standard_type = 'universal'
        elif standard_num in ['11', '12', '13', '14']:
            standard_type = 'sector'
        else:
            standard_type = 'topic'
        
        return {
            'standard_id': standard_id,
            'standard_num': standard_num,
            'standard_name': standard_name,
            'year': year,
            'standard_type': standard_type,
            'filename': filename
        }

### Xử lý text

In [6]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    with pdfplumber.open(pdf_path) as pdf:
        text = ""
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text


def clean_text(text: str) -> str:
    # Loại bỏ khoảng trắng
    text = re.sub(r'\s+', ' ', text)
    # Loại bỏ các xuống dòng thừa
    text = re.sub(r'\n+', '\n', text)
    # Loại bỏ khoảng trắng đầu/cuối
    text = text.strip()
    return text

### Xử lý REQUIREMENTS

In [7]:
def extract_requirements(text: str, standard_info: Dict) -> List[Dict]:
    """
    Trích xuất các REQUIREMENTS từ tiêu chuẩn GRI. Các tiêu chuẩn có cấu trúc phân cấp `DISCLOSURE --> REQUIREMENTS --> a, b, c... --> i, ii, iii...`
    1. Mỗi DISCLOSURE bắt đầu với "Disclosure {ID} {NAME}"
    2. Mỗi DISCLOSURE có phần REQUIREMENTS bắt đầu với "REQUIREMENTS"
    3. Các REQUIREMENTS chính được đánh dấu bằng chữ cái a., b., c., ...
    4. Các REQUIREMENTS phụ được đánh dấu bằng số La Mã i., ii., iii., ...
    5. Các REQUIREMENTS phụ xuất hiện sau một REQUIREMENTS chính kết thúc bằng ":"
    6. Xử lý các trường hợp đặc biệt và ngoại lệ trong cấu trúc văn bản. 
    """
    requirements = []
    
    ROMAN_NUMERALS = {'i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x', 'xi', 'xii', 'xiii', 'xiv', 'xv'}
    
    # Pattern để tìm disclosure
    disclosure_pattern = r'Disclosure\s+(\d+-\d+)\s+([^\n]+?)(?:\n|$)'
    
    # Tìm disclosure
    disclosure_matches = list(re.finditer(disclosure_pattern, text, re.IGNORECASE))
    
    if not disclosure_matches:
        return requirements
    
    # Xử lý từng disclosure
    for idx, match in enumerate(disclosure_matches):
        disclosure_id = match.group(1).strip()
        disclosure_name = match.group(2).strip()
        disclosure_name = re.sub(r'\s+', ' ', disclosure_name).strip()
        
        # Lấy phạm vi văn bản cho disclosure này
        start_pos = match.end()
        end_pos = disclosure_matches[idx + 1].start() if idx + 1 < len(disclosure_matches) else len(text)
        disclosure_text = text[start_pos:end_pos]
        
        # Tìm phần REQUIREMENTS
        req_start = re.search(
            r'(?:The\s+(?:reporting\s+)?organization\s+shall[:\s]+)?REQUIREMENTS',
            disclosure_text,
            re.IGNORECASE
        )
        
        if not req_start:
            continue
        
        # Trích xuất phần REQUIREMENTS
        req_text_start = req_start.end()
        req_section_end = re.search(
            r'\n(?:GUIDANCE|Guidance|RECOMMENDATIONS|Compilation requirements|Compiled GRI|^\d+\.)',
            disclosure_text[req_text_start:],
            re.IGNORECASE | re.MULTILINE
        )
        
        req_section = disclosure_text[req_text_start:req_text_start + req_section_end.start()] if req_section_end else disclosure_text[req_text_start:]
        
        # Tách thành các dòng để phân tích cấu trúc
        lines = req_section.split('\n')
        
        current_letter_req = None
        current_letter_text = []
        has_sub_reqs = False
        sub_requirements = []
        
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            
            if not line:
                i += 1
                continue
            
            # Kiểm tra xem đây có phải là REQUIREMENTS chính mới (a., b., c., ...)
            # NHƯNG KHÔNG PHẢI số La Mã (i., v., x., v.v.)
            letter_match = re.match(r'^([a-z])[\.\)]\s+(.+)$', line, re.IGNORECASE)
            
            # Kiểm tra xem có phải là số La Mã không
            is_roman = False
            if letter_match:
                potential_letter = letter_match.group(1).lower()
                if potential_letter in ROMAN_NUMERALS:
                    is_roman = True
                    letter_match = None
            
            if not letter_match:
                roman_match = re.match(r'^(i{1,3}|iv|v|vi{1,3}|ix|x|xi{1,3}|xiv|xv)[\.\)]\s+(.+)$', line, re.IGNORECASE)
                if roman_match:
                    is_roman = True
            else:
                roman_match = None
            
            if letter_match and not is_roman:
                # Lưu REQUIREMENTS chính trước đó nếu có
                if current_letter_req:
                    if has_sub_reqs and sub_requirements:
                        # Thêm REQUIREMENTS chính (văn bản trước các REQUIREMENTS phụ)
                        main_text = ' '.join(current_letter_text).strip()
                        # Loại bỏ dấu `:`
                        main_text = main_text.rstrip(':').strip()
                        if len(main_text) >= 10:
                            requirements.append({
                                'disclosure_id': disclosure_id,
                                'disclosure_name': disclosure_name,
                                'requirement_id': current_letter_req,
                                'requirement_text': main_text,
                                'hierarchy_level': 1,
                                'parent_requirement': None
                            })
                        # Thêm tất cả các REQUIREMENTS phụ
                        requirements.extend(sub_requirements)
                    else:
                        # Không có REQUIREMENTS phụ, thêm REQUIREMENTS hoàn chỉnh
                        full_text = ' '.join(current_letter_text).strip()
                        if len(full_text) >= 10:
                            requirements.append({
                                'disclosure_id': disclosure_id,
                                'disclosure_name': disclosure_name,
                                'requirement_id': current_letter_req,
                                'requirement_text': full_text,
                                'hierarchy_level': 1,
                                'parent_requirement': None
                            })
                
                # Bắt đầu REQUIREMENTS chính mới
                current_letter_req = letter_match.group(1).lower()
                current_letter_text = [letter_match.group(2)]
                has_sub_reqs = False
                sub_requirements = []
                
                # Kiểm tra xem dòng này có kết thúc bằng ":" không, điều này cho biết có các REQUIREMENTS phụ theo sau
                if letter_match.group(2).rstrip().endswith(':'):
                    has_sub_reqs = True
                
            elif is_roman and has_sub_reqs:
                # Đây là REQUIREMENTS phụ theo cấp bậc
                if roman_match:
                    roman_num = roman_match.group(1).lower()
                    sub_text = [roman_match.group(2)]
                elif letter_match:  # Nó được khớp như chữ cái nhưng thực ra là số La Mã
                    roman_num = letter_match.group(1).lower()
                    sub_text = [letter_match.group(2)]
                else:
                    i += 1
                    continue
                
                # Thu thập các dòng tiếp tục cho REQUIREMENTS phụ này
                j = i + 1
                while j < len(lines):
                    next_line = lines[j].strip()
                    if not next_line:
                        j += 1
                        continue
                    # Dừng lại nếu chúng ta gặp chữ cái tiếp theo hoặc số La Mã tiếp theo
                    next_letter = re.match(r'^([a-z])[\.\)]', next_line, re.IGNORECASE)
                    next_is_roman = False
                    if next_letter and next_letter.group(1).lower() in ROMAN_NUMERALS:
                        next_is_roman = True
                    elif re.match(r'^(i{1,3}|iv|v|vi{1,3}|ix|x|xi{1,3}|xiv|xv)[\.\)]', next_line, re.IGNORECASE):
                        next_is_roman = True
                    
                    if (next_letter and not next_is_roman) or next_is_roman:
                        break
                    sub_text.append(next_line)
                    j += 1
                
                i = j - 1
                
                sub_req_text = ' '.join(sub_text).strip()
                if len(sub_req_text) >= 10:
                    sub_requirements.append({
                        'disclosure_id': disclosure_id,
                        'disclosure_name': disclosure_name,
                        'requirement_id': f"{current_letter_req}.{roman_num}",
                        'requirement_text': sub_req_text,
                        'hierarchy_level': 2,
                        'parent_requirement': current_letter_req
                    })
            else:
                # Tiếp tục văn bản REQUIREMENTS chính hiện tại
                if current_letter_req:
                    current_letter_text.append(line)
            
            i += 1
        
        if current_letter_req:
            if has_sub_reqs and sub_requirements:
                main_text = ' '.join(current_letter_text).strip()
                main_text = main_text.rstrip(':').strip()
                if len(main_text) >= 10:
                    requirements.append({
                        'disclosure_id': disclosure_id,
                        'disclosure_name': disclosure_name,
                        'requirement_id': current_letter_req,
                        'requirement_text': main_text,
                        'hierarchy_level': 1,
                        'parent_requirement': None
                    })
                requirements.extend(sub_requirements)
            else:
                full_text = ' '.join(current_letter_text).strip()
                if len(full_text) >= 10:
                    requirements.append({
                        'disclosure_id': disclosure_id,
                        'disclosure_name': disclosure_name,
                        'requirement_id': current_letter_req,
                        'requirement_text': full_text,
                        'hierarchy_level': 1,
                        'parent_requirement': None
                    })
    
    # Thêm metadata
    claim_level = "in_accordance" if standard_info['standard_type'] == 'universal' else "with_reference"
    
    for req in requirements:
        req['requirement_type'] = 'shall'
        req['is_mandatory'] = True
        req['claim_level'] = claim_level
    
    return requirements


## Xử lý toàn bộ GRI standards

In [8]:
all_units = []

for pdf_path in tqdm(pdf_files, desc="Processing PDFs"):
    standard_info = parse_gri_filename(pdf_path.name)
    text = extract_text_from_pdf(pdf_path)

    if not text:
        continue

    requirements = extract_requirements(text, standard_info)

    if not requirements:
        continue

    for req in requirements:
        req_id_clean = req["requirement_id"].replace(".", "-")
        base_unit_id = (
            f"{standard_info['standard_id'].replace(' ', '')}-"
            f"{standard_info['year']}-"
            f"D{req['disclosure_id']}-"
            f"R{req_id_clean}"
        )

        unit_id = base_unit_id
        unit = {
            "unit_id": unit_id,
            "standard_id": standard_info["standard_id"],
            "standard_name": standard_info["standard_name"],
            "standard_type": standard_info["standard_type"],
            "disclosure_id": req["disclosure_id"],
            "disclosure_name": req["disclosure_name"],
            "requirement_id": req["requirement_id"],
            "requirement_text": req["requirement_text"][:8192],
            "requirement_type": req["requirement_type"],
            "claim_level": req["claim_level"],
            "is_mandatory": req["is_mandatory"],
            "year": standard_info["year"],
            "sector_applicability": (
                "all" if standard_info["standard_type"] != "sector" else standard_info["standard_name"]
            ),
            "hierarchy_level": req.get("hierarchy_level", 1),
            "parent_requirement": req.get("parent_requirement", None),
        }

        all_units.append(unit)


Processing PDFs: 100%|██████████| 38/38 [00:37<00:00,  1.00it/s]


In [10]:
all_units_df = pd.DataFrame(all_units)
all_units_df[:5]

,unit_id,standard_id,standard_name,standard_type,disclosure_id,disclosure_name,requirement_id,requirement_text,requirement_type,claim_level,is_mandatory,year,sector_applicability,hierarchy_level,parent_requirement
0,GRI101-2024-D101-1-Ra,GRI 101,Biodiversity,topic,101-1,Policies to halt and reverse,a,describe its policies or commitments to halt a...,shall,with_reference,True,2024,all,1,None
1,GRI101-2024-D101-1-Rb,GRI 101,Biodiversity,topic,101-1,Policies to halt and reverse,b,report the extent to which these policies or c...,shall,with_reference,True,2024,all,1,None
2,GRI101-2024-D101-1-Rc,GRI 101,Biodiversity,topic,101-1,Policies to halt and reverse,c,report the goals and targets to halt and rever...,shall,with_reference,True,2024,all,1,None
3,GRI101-2024-D101-2-Ra,GRI 101,Biodiversity,topic,101-2,Management of biodiversity impacts,a,report how it applies the mitigation hierarchy...,shall,with_reference,True,2024,all,1,None
4,GRI101-2024-D101-2-Ra-i,GRI 101,Biodiversity,topic,101-2,Management of biodiversity impacts,a.i,actions taken to avoid negative impacts on bio...,shall,with_reference,True,2024,all,2,a
